In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# For statistical tests and modeling
from scipy.stats import chi2_contingency, ttest_ind
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score



In [ ]:
# Load data
data = pd.read_csv('/kaggle/input/datasets/gourisharma32/ha-data/HA Data.csv')


In [ ]:
# Initial overview
print(data.shape)
print(data.info())
print(data.head())

In [ ]:
# Quick stats
data.describe(include='all')


In [ ]:
# Check for missing values
missing = data.isna().sum().sort_values(ascending=False)
print(missing)

In [ ]:
print(data['weight'].unique())

In [ ]:


# Replace '?' with NaN (if not already done)
data['weight'] = data['weight'].replace('?', np.nan)

def convert_weight(x):
    try:
        if pd.isna(x):
            return np.nan
        
        x = str(x).strip()
        
        # Case 1: '>200'
        if '>' in x:
            return 200   # you can also choose 210 or keep as NaN
        
        # Case 2: range like [0-25)
        x = x.replace('[','').replace(')','')
        low, high = x.split('-')
        return (int(low) + int(high)) / 2
    
    except:
        return np.nan

# Apply function
data['weight'] = data['weight'].apply(convert_weight)

In [ ]:
print(data['weight'].unique())

In [ ]:
def convert_age(x):
    try:
        x = str(x).strip()
        x = x.replace('[','').replace(')','')
        low, high = x.split('-')
        return (int(low) + int(high)) / 2
    except:
        return np.nan

data['age'] = data['age'].apply(convert_age)   #clean age

In [ ]:
data['medical_specialty'] = data['medical_specialty'].fillna('Unknown')

diag_cols = ['diag_1','diag_2','diag_3','diag_4','diag_5']
for col in diag_cols:
    data[col] = data[col].fillna('Unknown')

In [ ]:
# Convert readmitted into binary
data['readmitted'] = data['readmitted'].replace({
    '<30': 1,
    '>30': 0,
    'NO': 0
})

In [ ]:
cat_cols = ['race','gender','medical_specialty','change','diabetesMed']

for col in cat_cols:
    data[col] = data[col].astype('category')

In [ ]:
# Total previous visits
data['total_previous_visits'] = (
    data['number_outpatient'] +
    data['number_inpatient'] +
    data['number_emergency']
)

# Comorbidity score
data['comorbidity_score'] = data['number_diagnoses']

In [ ]:


num_cols = ['time_in_hospital','num_lab_procedures','num_medications']

for col in num_cols:
    sns.boxplot(x=data[col])
    plt.title(f'Boxplot of {col}')
    plt.show()

In [ ]:
print(data.isna().sum())   # should be minimal
print(data.dtypes)
print(data.head())

In [ ]:
data['medical_specialty'] = data['medical_specialty'].str.strip()
data['medical_specialty'] = data['medical_specialty'].replace('?', 'Unknown')

In [ ]:
data['diag_5'] = data['diag_5'].astype(str)

In [ ]:
data = data.drop(['X1','X2'], axis=1)

In [ ]:
med_cols = [f'X{i}' for i in range(3,26)]

for col in med_cols:
    data[col] = data[col].map({'Yes': 1, 'No': 0})

In [ ]:
data = data.drop('index', axis=1)

In [ ]:
# Fill missing medication values
med_cols = [f'X{i}' for i in range(3,26)]
data[med_cols] = data[med_cols].fillna(0)

# Convert to integer
data[med_cols] = data[med_cols].astype(int)

# Optional: drop patient_id
data = data.drop('patient_id', axis=1)

# Final check
print(data.isna().sum())
print(data.dtypes)

In [ ]:
readmit_rate = data['readmitted'].mean()
print(f"Overall Readmission Rate: {readmit_rate:.2f}")

In [ ]:
age_readmit = data.groupby('age')['readmitted'].mean()

import matplotlib.pyplot as plt
plt.figure(figsize=(10,5))
plt.plot(age_readmit.index, age_readmit.values)
plt.title('Readmission Rate by Age')
plt.xlabel('Age')
plt.ylabel('Readmission Rate')
plt.show()

Older patients show higher readmission rates, suggesting age-related vulnerability and need for targeted follow-up care.

In [ ]:
#readmission by gender and race

# Gender
sns.barplot(x='gender', y='readmitted', data=data)
plt.title('Readmission by Gender')
plt.show()

# Race
plt.figure(figsize=(10,5))
sns.barplot(x='race', y='readmitted', data=data)
plt.xticks(rotation=45)
plt.title('Readmission by Race')
plt.show()

Minor variations exist across gender and race groups, indicating possible disparities in care or access to healthcare services.

In [ ]:
sns.boxplot(x='readmitted', y='time_in_hospital', data=data)
plt.title('Length of Stay vs Readmission')
plt.show()   #length of saty vs readmission

Patients with longer hospital stays tend to have higher readmission rates, possibly indicating more severe health conditions

In [ ]:
sns.scatterplot(x='number_emergency', y='readmitted', data=data)
plt.title('Emergency Visits vs Readmission')
plt.show()

print("Correlation:", data['number_emergency'].corr(data['readmitted']))  #emergency visits vs readmission

In [ ]:
# Diabetes patients
sns.barplot(x='diabetesMed', y='readmitted', data=data)
plt.title('Diabetes Medication vs Readmission')
plt.show()

# Medication change
sns.barplot(x='change', y='readmitted', data=data)
plt.title('Medication Change vs Readmission')
plt.show()

Patients on diabetes medication and those with medication changes show higher readmission rates, suggesting complexity in disease management.


In [ ]:
sns.scatterplot(x='num_lab_procedures', y='readmitted', data=data)
plt.title('Lab Procedures vs Readmission')
plt.show()

sns.scatterplot(x='num_medications', y='readmitted', data=data)
plt.title('Medications vs Readmission')
plt.show()

Higher number of lab procedures and medications are associated with increased readmission risk, indicating severity of illness.

In [ ]:
sns.boxplot(x='readmitted', y='comorbidity_score', data=data)
plt.title('Comorbidity vs Readmission')
plt.show()

Patients with multiple diagnoses (higher comorbidity score) are significantly more likely to be readmitted.

In [ ]:
sns.scatterplot(x='total_previous_visits', y='readmitted', data=data)
plt.title('Previous Visits vs Readmission')
plt.show()

Frequent past hospital visits strongly correlate with readmission risk, making them key predictors.

In [ ]:
plt.figure(figsize=(12,8))
sns.heatmap(data.corr(numeric_only=True), cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()

In [ ]:
data.to_csv('hospital_cleaned.csv', index=False)